In [ ]:
# -*- coding: utf-8 -*-
"""project_cropipynb_cdac.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1dL94f_IDxqa7X003ojFyDdeInwMuyfR1
"""

!pip install timm

!pip install pytorch-tabnet

"""FULL CODE FOR CROP RECCOMENDATION **MODEL**"""

import torch
import torchvision
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Crop Recommendation Dataset
crop_rec_df = pd.read_csv('/content/new_agri.csv')
print("Crop Recommendation Dataset:")
print(crop_rec_df.head())
print(crop_rec_df.info())


# 3. Crop Recommendation with Weather Dataset
weather_crop_df = pd.read_csv('data_weather.csv')
print("\nCrop Recommendation with Weather Dataset:")
print(weather_crop_df.head())
print(weather_crop_df.info())

# Optional: Check for missing values in each dataset
print("\nMissing values in Crop Recommendation Dataset:\n", crop_rec_df.isnull().sum())
print("\nMissing values in Crop Recommendation with Weather Dataset:\n", weather_crop_df.isnull().sum())

weather_crop_df.rename(columns={
    'Soilcolor': 'Soil_Color',
    'Ph': 'Soil_pH',
    'K': 'Potassium',
    'P': 'Phosphorus',
    'N': 'Nitrogen',
    'Zn': 'Zinc',
    'S': 'Sulfur',

    'QV2M-W': 'Humidity_Winter',
    'QV2M-Sp': 'Humidity_Spring',
    'QV2M-Su': 'Humidity_Summer',
    'QV2M-Au': 'Humidity_Autumn',

    'T2M_MAX-W': 'Temp_Max_Winter',
    'T2M_MAX-Sp': 'Temp_Max_Spring',
    'T2M_MAX-Su': 'Temp_Max_Summer',
    'T2M_MAX-Au': 'Temp_Max_Autumn',

    'T2M_MIN-W': 'Temp_Min_Winter',
    'T2M_MIN-Sp': 'Temp_Min_Spring',
    'T2M_MIN-Su': 'Temp_Min_Summer',
    'T2M_MIN-Au': 'Temp_Min_Autumn',

    'PRECTOTCORR-W': 'Rainfall_Winter',
    'PRECTOTCORR-Sp': 'Rainfall_Spring',
    'PRECTOTCORR-Su': 'Rainfall_Summer',
    'PRECTOTCORR-Au': 'Rainfall_Autumn',

    'WD10M': 'Wind_Direction_10m',
    'GWETTOP': 'Topsoil_Moisture',
    'CLOUD_AMT': 'Cloud_Amount',
    'WS2M_RANGE': 'WindSpeed_Range_2m',
    'PS': 'Surface_Pressure',

    'label': 'Crop_Label'
}, inplace=True)

weather_crop_df.head()

import pandas as pd
import numpy as np

# Load original dataset
df = pd.read_csv("new_agri.csv")

# Ensure all soil_type columns are present
required_soil_cols = ['soil_0', 'soil_1', 'soil_2', 'soil_3']
for col in required_soil_cols:
    if col not in df.columns:
        df[col] = 0

# Define top 10 Bihar-specific crops with refined ranges
# These ranges are based on optimal conditions and typical agricultural practices in Bihar.

crop_settings = {

    'wheat': { # Rabi crop
        'temp': (18, 25),    # Cooler temperatures for vegetative growth and grain filling.
        'rain': (500, 750),  # Moderate rainfall, often supplemented by irrigation.
        'humid': (60, 85),   # Moderate humidity, too high risks fungal diseases.
        'ph': (6.0, 7.5),    # Slightly acidic to neutral/alkaline.
        'N': (100, 150),     # High N for tillering and grain formation.
        'P': (40, 60),       # Good P for root development.
        'K': (40, 60),       # Good K for disease resistance and grain quality.
        'soil_preference': [0, 3] # Alluvial, Loamy (well-drained fertile loams)
    },
    'sugarcane': { # Important cash crop
        'temp': (25, 35),    # Warm for vegetative, slightly cooler for ripening.
        'rain': (1000, 1500),# High rainfall, but requires good drainage; irrigation needed for dry periods.
        'humid': (70, 90),   # High humidity for tropical growth.
        'ph': (6.0, 7.5),    # Tolerant, prefers slightly acidic to neutral.
        'N': (150, 250),     # Very high N demand for biomass.
        'P': (50, 80),       # Good P for root and stalk vigor.
        'K': (80, 120),      # High K for sugar formation and disease resistance.
        'soil_preference': [0, 3] # Alluvial, Loamy (deep, fertile, well-drained)
    },
    'potato': { # Rabi crop
        'temp': (18, 25),    # Cooler temperatures for tuber formation.
        'rain': (500, 700),  # Moderate, well-distributed rainfall; irrigation often needed.
        'humid': (65, 85),   # Moderate humidity.
        'ph': (5.0, 6.0),    # Prefers slightly acidic soils.
        'N': (100, 150),     # High N for foliage, then shifted to tubers.
        'P': (60, 90),       # High P crucial for tuber development.
        'K': (120, 180),     # Very high K for tuber size, quality, and disease resistance.
        'soil_preference': [0, 1, 3] # Alluvial (sandy loam to loam), Sandy, Loamy (light, well-drained)
    },
    'litchi': { # Iconic fruit crop
        'temp': (24, 32),    # Optimal for growth and fruiting.
        'rain': (1200, 1500),# High rainfall during monsoon, less for flowering/fruiting.
        'humid': (75, 95),   # High humidity.
        'ph': (5.5, 6.5),    # Slightly acidic.
        'N': (70, 120),      # Higher N for vegetative growth and fruit size.
        'P': (30, 50),       # Moderate P.
        'K': (50, 80),       # High K for fruit quality and sweetness.
        'soil_preference': [0, 3] # Alluvial, Loamy (deep, well-drained)
    },
    'makhana': { # Unique aquatic crop
        'temp': (25, 35),    # Warm temperatures for aquatic growth.
        'rain': (1200, 2000),# Very high water requirement, often standing water.
        'humid': (80, 100),  # Very high humidity due to aquatic environment.
        'ph': (6.0, 7.5),    # Slightly acidic to neutral/slightly alkaline.
        'N': (50, 100),      # Moderate N, often supplemented by organic matter.
        'P': (25, 45),       # Moderate P.
        'K': (35, 60),       # Moderate K.
        'soil_preference': [0, 2] # Alluvial, Clay (heavy, water-retentive pond beds)
    },
    'tomato': { # Primarily Rabi (winter) crop
        'temp': (20, 30),    # Optimal for fruit set; sensitive to heat over 35°C.
        'rain': (600, 900),  # Moderate, well-distributed rainfall; sensitive to waterlogging.
        'humid': (60, 85),   # Moderate humidity; high humidity increases disease.
        'ph': (6.0, 7.0),    # Slightly acidic to neutral.
        'N': (80, 150),      # High N for vegetative growth and fruiting.
        'P': (50, 90),       # Good P for flowering and fruit development.
        'K': (80, 150),      # High K for fruit quality, size, and disease resistance.
        'soil_preference': [0, 3] # Alluvial, Loamy (well-drained sandy loam to clay loam)
    },
    'onion': { # Rabi crop
        'temp': (20, 30),    # Optimal for growth, cooler for bulbing.
        'rain': (600, 800),  # Moderate, consistent moisture during growth, dry for maturity.
        'humid': (60, 80),   # Moderate humidity.
        'ph': (6.0, 7.0),    # Slightly acidic to neutral.
        'N': (80, 120),      # High N for leafy growth.
        'P': (40, 60),       # Moderate P for root and bulb development.
        'K': (80, 100),      # High K for bulb size and storage quality.
        'soil_preference': [0, 1, 3] # Alluvial (sandy loam to loam), Sandy, Loamy (well-drained)
    }
}

# Generate synthetic data
augmented = []
for crop, params in crop_settings.items():
    for _ in range(100): # Generate 100 samples per crop
        # Select soil type based on preference
        soil_type = np.random.choice(params['soil_preference'])
        soil_vector = [1 if i == soil_type else 0 for i in range(4)] # Assuming 4 soil types (0-3)

        augmented.append([
            np.random.uniform(*params['N']),
            np.random.uniform(*params['P']),
            np.random.uniform(*params['K']),
            np.random.uniform(*params['temp']),
            np.random.uniform(*params['humid']),
            np.random.uniform(*params['ph']),
            np.random.uniform(*params['rain']),
            crop,
            *soil_vector
        ])

aug_df = pd.DataFrame(augmented, columns=[
    'N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label',
    'soil_0', 'soil_1', 'soil_2', 'soil_3'
])

# Concatenate and save
df_aug = pd.concat([df, aug_df], ignore_index=True)
df_aug.to_csv("augmented_bihar_top10_crops.csv", index=False) # Changed filename again
print("✅ Dataset successfully augmented and saved as 'augmented_bihar_top10_crops.csv'")
print("🔢 New shape:", df_aug.shape)

df_aug['label'].unique()

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import random

# --------------------
# 1. Load & Preprocess Data
# --------------------
df = pd.read_csv('augmented_bihar_top10_crops.csv')  # Replace with your actual file

# Encode crop labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

# Add synthetic soil_type if missing
if 'soil_type' not in df.columns:
    np.random.seed(42)
    df['soil_type'] = np.random.randint(0, 4, size=len(df))

# One-hot encode soil type
soil_dummies = pd.get_dummies(df['soil_type'], prefix='soil')
df = pd.concat([df.drop('soil_type', axis=1), soil_dummies], axis=1)

# Split features and labels
X = df.drop('label', axis=1).values
y = df['label'].values

# Normalize only numeric features (assume first 7 are numerical)
scaler = StandardScaler()
X_num = scaler.fit_transform(X[:, :7])
X_cat = X[:, 7:]
X_final = np.concatenate([X_num, X_cat], axis=1).astype(np.float32)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

# --------------------
# 2. Dataset & DataLoader
# --------------------
class CropDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = CropDataset(X_train, y_train)
test_dataset = CropDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# --------------------
# 3. Define Model
# --------------------
class CropNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(CropNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.dropout = nn.Dropout(0.5) # <--- Changed this
        self.out = nn.Linear(32, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        return self.out(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CropNN(input_size=X_final.shape[1], num_classes=len(np.unique(y))).to(device)

# --------------------
# 4. Training
# --------------------
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

num_epochs = 30
train_losses = []
val_accuracies = []

best_val_acc = 0
patience = 5
counter = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * features.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader.dataset)
    train_acc = 100 * correct / total
    train_losses.append(train_loss)

    # Evaluate on test set
    model.eval()
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            _, predicted = torch.max(outputs, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    val_acc = 100 * correct_test / total_test
    val_accuracies.append(val_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%")

    # Early Stopping Check
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0  # reset counter if validation improves
        best_model_state = model.state_dict()  # save best model weights
    else:
        counter += 1
        if counter >= patience:
            print(f"\n🛑 Early stopping triggered at epoch {epoch+1}. Best Val Acc: {best_val_acc:.2f}%")
            break

# Load best model
model.load_state_dict(best_model_state)
# --------------------
# 5. Plot Learning Curve
# --------------------
plt.plot(train_losses, label='Train Loss')
plt.plot(val_accuracies, label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Loss / Accuracy')
plt.legend()
plt.title("Training Loss & Validation Accuracy")
plt.grid(True)
plt.show()

# --------------------
# 6. Final Evaluation
# --------------------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for features, labels in test_loader:
        features = features.to(device)
        outputs = model(features)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print("\nFinal Test Accuracy: {:.2f}%".format(accuracy_score(all_labels, all_preds) * 100))
print(classification_report(all_labels, all_preds, target_names=le.classes_))

print(f"Epoch [{epoch+1}/{num_epochs}] ✅ Summary:")
print(f"    🔹 Train Loss: {train_loss:.4f}")
print(f"    🔹 Train Accuracy: {train_acc:.2f}%")
print(f"    🔹 Validation Accuracy: {val_acc:.2f}%")

# Check for overfitting (optional rule of thumb)
if (train_acc - val_acc) > 5:
    print("⚠️  Possible Overfitting: Training accuracy is significantly higher than validation accuracy.")

import numpy as np
import torch

# Define correct soil type mapping (8 total)
soil_type_map_full = {
    'alluvial': 0,
    'black':    1,
    'red':      2,
    'laterite': 3
}

# Function to return one-hot for 8-class soil encoding
def get_soil_one_hot(index, total_classes=8):
    one_hot = np.zeros(total_classes)
    one_hot[index] = 1
    return one_hot

# Auto NPK ranges per soil
def get_default_npk_for_soil(soil_type):
    soil_type = soil_type.lower()
    if soil_type == 'alluvial':
        return 90, 35, 45
    elif soil_type == 'black':
        return 100, 40, 50
    elif soil_type == 'red':
        return 50, 25, 70
    elif soil_type == 'laterite':
        return 40, 22, 60
    else:
        raise ValueError("Invalid soil type")

# --- 1. Get User Input ---
soil = input("🟤 Enter soil type (alluvial / black / red / laterite): ").strip().lower()
if soil not in soil_type_map_full:
    raise ValueError("Invalid soil type.")

use_auto_npk = input("🔄 Auto-fill NPK based on soil type? (yes/no): ").strip().lower()
if use_auto_npk == 'yes':
    N, P, K = get_default_npk_for_soil(soil)
else:
    N = float(input("Enter Nitrogen (N): "))
    P = float(input("Enter Phosphorus (P): "))
    K = float(input("Enter Potassium (K): "))

temperature = float(input("🌡️ Enter Temperature (°C): "))
humidity = float(input("💧 Enter Humidity (%): "))
ph = float(input("🧪 Enter soil pH: "))
rainfall = float(input("🌧️ Enter Rainfall (mm): "))

# --- 2. Format & preprocess input ---
custom_raw = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
custom_num = scaler.transform(custom_raw)

soil_index = soil_type_map_full[soil]
soil_one_hot = get_soil_one_hot(soil_index, total_classes=8).reshape(1, -1)

custom_input = np.concatenate([custom_num, soil_one_hot], axis=1).astype(np.float32)
custom_tensor = torch.tensor(custom_input, dtype=torch.float32).to(device)

# --- 3. Prediction using PyTorch Model ---
model.eval()
with torch.no_grad():
    output = model(custom_tensor)
    probs = torch.softmax(output, dim=1).cpu().numpy()[0]
    top_indices = probs.argsort()[::-1][:3]

# --- 4. Display Results ---
print("\n📈 Prediction Results:")
for idx in top_indices:
    print(f"  {le.classes_[idx]}: {probs[idx]:.4f}")
print(f"✅ Best Suited Crop: {le.classes_[top_indices[0]]}")